In [1]:
from google.cloud import bigquery
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

client = bigquery.Client()

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

query = f"""
SELECT content_clean, label
FROM `{project_id}.{dataset_id}.reviews_cleaned`
"""

sent_df = client.query(query).to_dataframe()

print(sent_df.shape)
sent_df.head()


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(50000, 2)


,content_clean,label
0,"God save the Queen and Sharpe, the English Rambo",0
1,DO NOT BUY THIS. It is an insult to the human ...,0
2,Length:: 3:49 MinsFahrenheit 451Fahrenheit 451...,0
3,I'd rather shoot myself than lean from this bo...,0
4,I'm surprised that nobody was ever told to use...,0


In [2]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Loaded model:", model_name, "on device:", device)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loaded model: distilbert-base-uncased-finetuned-sst-2-english on device: cuda


In [3]:
import numpy as np
from tqdm import tqdm

batch_size = 64
sentences = sent_df["content_clean"].tolist()
preds = []

model.eval()

with torch.no_grad():
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        
        # Predicted label: 1 = positive, 0 = negative
        batch_preds = probabilities.argmax(dim=1).cpu().numpy()
        preds.extend(batch_preds)

# Add predictions to DataFrame
sent_df["sentiment_pred"] = preds

sent_df.head()


100%|██████████| 782/782 [01:37<00:00,  8.03it/s]


,content_clean,label,sentiment_pred
0,"God save the Queen and Sharpe, the English Rambo",0,0
1,DO NOT BUY THIS. It is an insult to the human ...,0,0
2,Length:: 3:49 MinsFahrenheit 451Fahrenheit 451...,0,1
3,I'd rather shoot myself than lean from this bo...,0,0
4,I'm surprised that nobody was ever told to use...,0,1


In [5]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_id = "sentiment_scored"

table_full_id = f"{project_id}.{dataset_id}.{table_id}"

client = bigquery.Client()

# Save to parquet for load
sent_df.to_parquet("data/sentiment_scored.parquet", index=False)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

with open("data/sentiment_scored.parquet", "rb") as f:
    job = client.load_table_from_file(
        f, table_full_id, job_config=job_config
    )

job.result()

# Verify
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [c.name for c in table.schema])


Loaded rows: 50000
Columns: ['content_clean', 'label', 'sentiment_pred']
